# Week 03 — Statistical Approach to EDA

**Spatial Data Science with Python · MSc GIS / AGMS 1333**  
**Official Table 4 topic:** Understanding Exploratory Data Analysis · Statistics, data cleaning, missingness, correlation

**Learning outcome.** Summarise June surface-temperature distribution and canopy association.

**What you will implement:** Sample SD, IQR, Pearson and Spearman, plots.  
**Mathematical model:** s = sqrt(Σ(yᵢ−ȳ)²/(n−1)); IQR = Q₃−Q₁.

> **Data provenance:** The Urban Teaching Grid is a **synthetic** 12 × 12 grid at 250-m cell spacing. Its temperatures and facilities are simulated; do not present its numerical outputs as field measurements. All distance calculations in the labs use the projected coordinates EPSG:32647 where relevant.

**How to run.** In Colab, upload this notebook from *File → Upload notebook*. Run each cell top-to-bottom. The setup cell below will ask for `DATA_ONLY.zip` only if the data folder is absent. In JupyterLab, extract the complete ZIP and open this notebook in `01_Core_Labs/`.


## 1. Prepare the environment and locate the data

The same setup works in JupyterLab and Google Colab. It selects an existing local data directory, or prompts for the packaged dataset in Colab. `assert` fails early if the archive is incomplete.

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local Jupyter: data/ or ../data. Colab: upload DATA_ONLY.zip once per runtime.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA = next((p for p in candidates if (p / 'cells.csv').is_file()), None)
if DATA is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Cannot find data/. Extract the course ZIP and keep the data folder with the notebooks.') from exc
    print('Select DATA_ONLY.zip from the downloaded course package:')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
    if zip_name is None:
        raise FileNotFoundError('Please upload DATA_ONLY.zip and run this cell again.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(Path.cwd())
    DATA = Path('data')
assert (DATA / 'metadata.json').is_file(), 'Data package incomplete.'
print('Dataset folder:', DATA.resolve())
print('Synthetic teaching data: not field observations.')


## 2. Key concepts and expected data

The analysis follows **input → cleaning → computation → validation → interpretation**. Always inspect observation keys, coordinate units, missing values, and the intended study period before fitting a model.

**Mathematical foundation:** s = sqrt(Σ(yᵢ−ȳ)²/(n−1)); IQR = Q₃−Q₁

**Method family:** Sample SD, IQR, Pearson and Spearman, plots.

## 3. Main computation

Run this cell after the setup. The block calculates the principal statistics or fits the weekly model. The original course code is retained so results remain aligned with the slide examples.

In [ ]:
cells=pd.read_csv(DATA/'cells.csv');obs=pd.read_csv(DATA/'observations_clean.csv')
june=cells.merge(obs[obs.date=='2025-06-01'],on='cell_id')
y=june.lst_c.dropna()
q1,q3=y.quantile([.25,.75]);print({'n':len(y),'mean':y.mean(),'median':y.median(),'sample_sd':y.std(ddof=1),'IQR':q3-q1})
print('Missing canopy:',june.canopy_frac.isna().sum())

### Reading the main code

**Descriptive statistics.** `dropna()` excludes unavailable temperatures, `quantile` extracts quartiles, and `std(ddof=1)` computes sample SD.

- `pd.read_csv` — Reads CSV text into a pandas DataFrame. Inspect head() and column types after loading.
- `merge` — Joins tables on a declared key; validate cardinality to avoid duplicate matches.
- `quantile` — Returns a data percentile for thresholds or distribution summaries.

**Verification questions:** Which variables contain the inputs? What new object does the cell create? What numerical result should be checked before continuing?

## 4. Map, plot, or evaluate the result

The final block turns results into a labelled graphic or output artifact. Keep units in axes/legends, and distinguish model predictions from observations.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].hist(y,bins=12,color='steelblue');ax[0].axvline(y.mean(),ls='--',color='red');ax[0].set_title('Temperature distribution')
pairs=june[['canopy_frac','lst_c']].dropna()
ax[1].scatter(pairs.canopy_frac,pairs.lst_c,alpha=.7);ax[1].set(xlabel='Canopy fraction',ylabel='LST (°C)',title='Correlation, not causation')
plt.show();print('Pearson',pairs.corr().iloc[0,1]);print('Spearman',pairs.corr(method='spearman').iloc[0,1])

### Reading the output code

**Plots.** The histogram describes one variable; the canopy–LST scatter uses complete pairs and may have fewer cells.

- `plt.` — Creates a figure; title, axes, and legend are part of the analytical interpretation.

**Pearson/Spearman.** The first summarizes linear association and the second monotonic rank association. Neither demonstrates causality.

## 5. Critical interpretation

### Interpret your results

Complete the following short report **after** running the notebook:

1. **Research question:** What geographic question does this notebook address?
2. **Observation unit and period:** What does one row/pixel represent? What is its date and CRS?
3. **Method:** Why is this method suitable, and what assumptions does it make?
4. **Results:** Report two numerical outputs or figure patterns with units.
5. **Validity:** State one data limitation and one model/evaluation limitation.
6. **Extension:** Modify one parameter, compare results, and describe what changed.


## 6. Questions and independent practice

1. Recompute summary statistics after excluding one outlier; discuss whether exclusion is scientifically justified.
2. What would happen if the spatial unit, study date, or CRS changed?
3. Is this analysis descriptive, inferential, predictive, or a decision-support example? Justify your answer.
4. Which assumption should be tested before use in a real project?

**Next step:** Open the matching advanced notebook in `02_Advanced_Experiments/` for four short method variations.